# Quick MACE-MH-1 MD Check

This notebook reads one generated `POSCAR_*`, reuses the initial positions and velocities stored in that POSCAR, attaches the local MACE-MH-1 model, and runs a short constant-energy NVE trajectory to check whether the generated initial condition is reasonable.

## Notes

- The notebook is intended for quick validation, not production MD.
- The NVE trajectory starts from the positions and velocities read from the POSCAR.
- By default the notebook uses the local model file `model/mace-mh-1.model`.
- A `tqdm` progress bar is shown during the MD run.
- The final trajectory is animated directly inside the notebook, not only shown as static snapshots.
- A final optional `nglview` cell lets you rotate and inspect the trajectory interactively with the mouse.

In [ ]:
import json
import os
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "mplconfig_qct_poscar_generator"))

import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
from IPython.display import HTML, display
from tqdm.auto import tqdm

from ase import units
from ase.io import Trajectory, read, write
from ase.md.verlet import VelocityVerlet
from ase.visualize.plot import plot_atoms

try:
    import torch
except ImportError as exc:
    raise ImportError("PyTorch is required for MACE. Install it in this environment first.") from exc

try:
    from mace.calculators import mace_mp
except ImportError as exc:
    raise ImportError("mace-torch is required for this notebook. Install it with pip install mace-torch.") from exc

try:
    import nglview as nv
except ImportError:
    nv = None

In [ ]:
CONFIG = {
    "poscar_glob": "outputs/qct_poscars/POSCAR_*",
    "poscar_index": 0,
    "molecule_vasprun": "inputs/molecule/vasprun.xml",
    "modes_npz": "inputs/molecule/vibrational_modes.npz",
    "model_path": "model/mace-mh-1.model",
    "head": "omat_pbe",
    "default_dtype": "float64",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "dispersion": False,
    "timestep_fs": 1.0,
    "n_steps": 1000,
    "log_interval": 5,
    "traj_write_interval": 1,
    "snapshot_stride": 20,
    "mode_plot_stride": 1,
    "mode_plot_max_modes": 6,
    "animation_stride": 5,
    "animation_interval_ms": 120,
    "initialize_velocities_if_missing": False,
    "fallback_temperature_K": 300.0,
    "output_dir": "outputs/mace_md_check",
    "trajectory_name": "quick_check.traj",
    "save_xyz": False,
    "xyz_name": "quick_check.xyz",
    "xyz_format": "extxyz",
}

print(json.dumps(CONFIG, indent=2))

In [ ]:
def choose_poscar(poscar_glob: str, poscar_index: int) -> Path:
    paths = sorted(Path().glob(poscar_glob))
    if not paths:
        raise FileNotFoundError(f"No POSCAR found for pattern: {poscar_glob}")
    if poscar_index < 0 or poscar_index >= len(paths):
        raise IndexError(f"poscar_index={poscar_index} is outside [0, {len(paths) - 1}]")
    return paths[poscar_index]


def build_mace_calculator(config: dict):
    model_path = Path(config["model_path"]).resolve()
    if not model_path.exists():
        raise FileNotFoundError(f"MACE model file not found: {model_path}")
    return mace_mp(
        model=str(model_path),
        head=config["head"],
        device=config["device"],
        default_dtype=config["default_dtype"],
        dispersion=config["dispersion"],
    )


def ensure_velocities(atoms, config: dict) -> None:
    vel = atoms.get_velocities()
    if vel is not None:
        return
    if not config["initialize_velocities_if_missing"]:
        raise ValueError(
            "No velocities were found in the POSCAR. Either regenerate the POSCAR with velocities or set initialize_velocities_if_missing=True."
        )
    from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

    MaxwellBoltzmannDistribution(atoms, temperature_K=config["fallback_temperature_K"])


def summarize_atoms(atoms) -> dict:
    vel = atoms.get_velocities()
    return {
        "natoms": len(atoms),
        "formula": atoms.get_chemical_formula(),
        "cell_lengths_A": atoms.cell.lengths().tolist(),
        "pbc": atoms.pbc.tolist(),
        "has_velocities": vel is not None,
        "max_velocity_A_fs": None if vel is None else float(np.abs(vel * units.fs).max()),
    }


def export_trajectory_xyz(traj_path: Path, config: dict) -> Path | None:
    if not config.get("save_xyz", False):
        return None

    xyz_format = config.get("xyz_format", "extxyz")
    if xyz_format not in {"xyz", "extxyz"}:
        raise ValueError("xyz_format must be 'xyz' or 'extxyz'.")

    xyz_path = Path(config["output_dir"]) / config["xyz_name"]
    frames = read(traj_path, index=":")
    if not isinstance(frames, list):
        frames = [frames]
    write(xyz_path, frames, format=xyz_format)
    return xyz_path


def plot_structure_views(atoms, title: str = "Initial structure"):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    rotations = ["0x,0y,0z", "90x,0y,0z", "0x,90y,0z"]
    labels = ["Top view", "Side view xz", "Side view yz"]
    for ax, rot, label in zip(axes, rotations, labels):
        plot_atoms(atoms, ax, rotation=rot, radii=0.5)
        ax.set_title(label)
        ax.set_axis_off()
    fig.suptitle(title)
    fig.tight_layout()
    return fig


def run_nve(atoms, config: dict):
    output_dir = Path(config["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)
    traj_path = output_dir / config["trajectory_name"]
    if traj_path.exists():
        traj_path.unlink()

    dyn = VelocityVerlet(atoms, timestep=config["timestep_fs"] * units.fs)
    trajectory = Trajectory(str(traj_path), "w", atoms)

    history = {
        "step": [],
        "time_fs": [],
        "epot_eV": [],
        "ekin_eV": [],
        "etot_eV": [],
        "temperature_K": [],
    }

    def log_step(write_frame: bool = True):
        step = dyn.nsteps
        epot = atoms.get_potential_energy()
        ekin = atoms.get_kinetic_energy()
        temp = atoms.get_temperature()
        history["step"].append(step)
        history["time_fs"].append(step * config["timestep_fs"])
        history["epot_eV"].append(epot)
        history["ekin_eV"].append(ekin)
        history["etot_eV"].append(epot + ekin)
        history["temperature_K"].append(temp)
        if write_frame:
            trajectory.write(atoms)

    log_step(write_frame=True)
    for _ in tqdm(range(config["n_steps"]), desc="NVE steps"):
        dyn.run(1)
        write_frame = (dyn.nsteps % config["traj_write_interval"] == 0)
        if write_frame or (dyn.nsteps % config["log_interval"] == 0):
            log_step(write_frame=write_frame)

    trajectory.close()
    xyz_path = export_trajectory_xyz(traj_path, config)
    return history, traj_path, xyz_path


def plot_md_diagnostics(history: dict):
    t = np.array(history["time_fs"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(t, history["epot_eV"], label="Epot")
    axes[0].plot(t, history["ekin_eV"], label="Ekin")
    axes[0].plot(t, history["etot_eV"], label="Etot")
    axes[0].set_xlabel("Time (fs)")
    axes[0].set_ylabel("Energy (eV)")
    axes[0].legend()
    axes[0].set_title("Energy conservation check")

    axes[1].plot(t, history["temperature_K"], color="tab:red")
    axes[1].set_xlabel("Time (fs)")
    axes[1].set_ylabel("Temperature (K)")
    axes[1].set_title("Instantaneous temperature")

    fig.tight_layout()
    return fig


def plot_component_temperatures(traj_path: Path, molecule_natoms: int, timestep_fs: float, stride: int = 1):
    frames = read(traj_path, index=":")
    if not isinstance(frames, list):
        frames = [frames]
    frames = frames[::stride]
    if not frames:
        raise ValueError("No trajectory frames available for temperature analysis.")
    if molecule_natoms <= 0 or molecule_natoms >= len(frames[0]):
        raise ValueError("molecule_natoms must be between 1 and natoms - 1.")

    times_fs = np.arange(len(frames), dtype=float) * timestep_fs * stride
    total_temp = []
    surface_temp = []
    molecule_temp = []
    for atoms_i in frames:
        total_temp.append(atoms_i.get_temperature())
        surface_temp.append(atoms_i[:-molecule_natoms].get_temperature())
        molecule_temp.append(atoms_i[-molecule_natoms:].get_temperature())

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(times_fs, total_temp, label="Total", color="tab:red", linewidth=1.8)
    ax.plot(times_fs, surface_temp, label="Surface only", color="tab:blue", linewidth=1.5)
    ax.plot(times_fs, molecule_temp, label="Molecule only", color="tab:green", linewidth=1.5)
    ax.set_xlabel("Time (fs)")
    ax.set_ylabel("Temperature (K)")
    ax.set_title("Instantaneous temperature by subsystem")
    ax.legend()
    fig.tight_layout()
    return fig


THZ_TO_CM1 = 33.35640951981521
EV_TO_J = 1.602176634e-19
AMU_TO_KG = 1.66053906660e-27
ANG_TO_M = 1.0e-10
FS_TO_S = 1.0e-15
C_CM_S = 2.99792458e10
LAMBDA_TO_OMEGA2_FS2 = (EV_TO_J / (AMU_TO_KG * ANG_TO_M**2)) * (FS_TO_S**2)
KCONV = AMU_TO_KG * (ANG_TO_M / FS_TO_S)**2 / EV_TO_J


def freq_cm1_to_omega_fs(freq_cm1: float) -> float:
    omega_si = 2.0 * np.pi * C_CM_S * freq_cm1
    return omega_si * FS_TO_S


def is_linear_molecule(atoms, tol: float = 1e-8) -> bool:
    pos = atoms.get_positions()
    pos_centered = pos - pos.mean(axis=0)
    _, s, _ = np.linalg.svd(pos_centered, full_matrices=False)
    if len(s) < 2:
        return True
    return s[1] < tol


def get_vibrational_mode_info(atoms) -> dict:
    nat = len(atoms)
    ndof = 3 * nat
    linear = is_linear_molecule(atoms)
    n_zero = 5 if linear else 6
    return {"nat": nat, "ndof": ndof, "linear": linear, "n_zero": n_zero, "n_vib": ndof - n_zero}


def normalize_isolated_molecule(atoms):
    mol = atoms.copy()
    ref = mol.positions[0].copy()
    new_pos = np.zeros((len(mol), 3))
    new_pos[0] = ref
    for i in range(1, len(mol)):
        new_pos[i] = ref + mol.get_distance(0, i, mic=True, vector=True)
    mol.set_positions(new_pos)
    mol.positions -= mol.get_center_of_mass()
    mol.set_pbc(False)
    mol.set_cell(np.zeros((3, 3)))
    return mol


def load_mode_basis(molecule_vasprun: str, modes_npz: str):
    ref_atoms = normalize_isolated_molecule(read(molecule_vasprun, index=-1, format="vasp-xml"))
    mode_data = np.load(modes_npz)
    freqs_cm1 = np.asarray(mode_data["freqs_cm1"], dtype=float)
    eigvecs_cart = np.asarray(mode_data["eigvecs_cart"], dtype=float)
    eig_mass, masses_dof = build_mass_weighted_modes(ref_atoms, eigvecs_cart)
    return {
        "reference_atoms": ref_atoms,
        "freqs_cm1": freqs_cm1,
        "eig_mass": eig_mass,
        "masses_dof": masses_dof,
        "mode_info": get_vibrational_mode_info(ref_atoms),
    }


def reshape_eigvecs(eig: np.ndarray, nat: int) -> np.ndarray:
    if eig.ndim == 3:
        n_modes, n_atoms, n_dir = eig.shape
        if n_atoms != nat or n_dir != 3:
            raise ValueError("The eigenvector dimensions do not match the molecule.")
        eig = eig.reshape(n_modes, n_atoms * n_dir)
    if eig.ndim != 2:
        raise ValueError("Unsupported eigenvector format.")
    n_modes, ndof = eig.shape
    if ndof != 3 * nat or n_modes != 3 * nat:
        raise ValueError("Expected a full 3N x 3N mode set.")
    return eig.T


def build_mass_weighted_modes(atoms, eig_cart: np.ndarray):
    nat = len(atoms)
    eig_matrix = reshape_eigvecs(eig_cart, nat)
    masses = np.repeat(atoms.get_masses(), 3)
    sqrt_m = np.sqrt(masses)
    eig_mass = eig_matrix / sqrt_m[:, None]
    return eig_mass, masses


def project_normal_coordinates(delta: np.ndarray, eig_mass: np.ndarray, masses_dof: np.ndarray) -> np.ndarray:
    delta_flat = np.asarray(delta, dtype=float).reshape(-1)
    return eig_mass.T @ (masses_dof * delta_flat)


def extract_isolated_molecule_frame(system_atoms, molecule_natoms: int):
    mol = system_atoms[-molecule_natoms:].copy()
    ref = mol.positions[0].copy()
    new_pos = np.zeros((len(mol), 3))
    new_pos[0] = ref
    for i in range(1, len(mol)):
        new_pos[i] = ref + mol.get_distance(0, i, mic=True, vector=True)
    mol.set_positions(new_pos)
    mol.set_pbc(False)
    mol.set_cell(np.zeros((3, 3)))
    return mol


def weighted_kabsch_rotation(mobile: np.ndarray, target: np.ndarray, weights: np.ndarray) -> np.ndarray:
    cov = mobile.T @ (weights[:, None] * target)
    u, _, vt = np.linalg.svd(cov)
    rot = u @ vt
    if np.linalg.det(rot) < 0:
        u[:, -1] *= -1.0
        rot = u @ vt
    return rot


def remove_rigid_motion(positions: np.ndarray, velocities: np.ndarray, masses: np.ndarray):
    total_mass = masses.sum()
    pos_com = (masses[:, None] * positions).sum(axis=0) / total_mass
    vel_com = (masses[:, None] * velocities).sum(axis=0) / total_mass
    rel = positions - pos_com
    vel_rel = velocities - vel_com
    inertia = np.zeros((3, 3))
    ang_momentum = np.zeros(3)
    for mass, r, v in zip(masses, rel, vel_rel):
        inertia += mass * (np.dot(r, r) * np.eye(3) - np.outer(r, r))
        ang_momentum += mass * np.cross(r, v)
    try:
        omega = np.linalg.solve(inertia, ang_momentum)
    except np.linalg.LinAlgError:
        omega = np.zeros(3)
    vib_vel = vel_rel - np.cross(omega[None, :], rel)
    return rel, vib_vel


def analyze_molecular_vibrations(traj_path: Path, mode_basis: dict, molecule_natoms: int, timestep_fs: float, stride: int = 1):
    frames = read(traj_path, index=":")
    if not isinstance(frames, list):
        frames = [frames]
    frames = frames[::stride]
    if not frames:
        raise ValueError("No trajectory frames available for vibrational analysis.")

    ref_atoms = mode_basis["reference_atoms"]
    ref_pos = ref_atoms.get_positions()
    masses = ref_atoms.get_masses()
    freqs_cm1 = mode_basis["freqs_cm1"]
    eig_mass = mode_basis["eig_mass"]
    masses_dof = mode_basis["masses_dof"]
    mode_info = mode_basis["mode_info"]
    n_zero = mode_info["n_zero"]
    vib_indices = np.arange(n_zero, len(freqs_cm1))

    times_fs = np.arange(len(frames), dtype=float) * timestep_fs * stride
    mode_kinetic = np.zeros((len(frames), len(vib_indices)))
    mode_potential = np.zeros((len(frames), len(vib_indices)))

    for frame_idx, system_atoms in enumerate(frames):
        mol = extract_isolated_molecule_frame(system_atoms, molecule_natoms)
        positions = mol.get_positions()
        velocities = mol.get_velocities()
        if velocities is None:
            raise ValueError("Trajectory frame is missing velocities required for mode analysis.")
        velocities = velocities * units.fs

        rel_pos, rel_vel = remove_rigid_motion(positions, velocities, masses)
        rot = weighted_kabsch_rotation(rel_pos, ref_pos, masses)
        pos_aligned = rel_pos @ rot
        vel_aligned = rel_vel @ rot
        _, vel_vib = remove_rigid_motion(pos_aligned, vel_aligned, masses)

        delta_pos = pos_aligned - ref_pos
        q = project_normal_coordinates(delta_pos, eig_mass, masses_dof)
        qdot = project_normal_coordinates(vel_vib, eig_mass, masses_dof)
        full_kinetic = 0.5 * (qdot**2) * KCONV
        full_potential = np.zeros_like(full_kinetic)
        for local_idx, mode_idx in enumerate(vib_indices):
            freq = freqs_cm1[mode_idx]
            if freq <= 0:
                continue
            omega_fs = freq_cm1_to_omega_fs(freq)
            lam = (omega_fs ** 2) / LAMBDA_TO_OMEGA2_FS2
            full_potential[mode_idx] = 0.5 * lam * (q[mode_idx] ** 2)
            mode_kinetic[frame_idx, local_idx] = full_kinetic[mode_idx]
            mode_potential[frame_idx, local_idx] = full_potential[mode_idx]

    mode_total = mode_kinetic + mode_potential
    total_vibrational_energy = mode_total.sum(axis=1)
    relative_mode_energy = np.divide(
        mode_total,
        total_vibrational_energy[:, None],
        out=np.zeros_like(mode_total),
        where=total_vibrational_energy[:, None] > 0.0,
    )
    return {
        "times_fs": times_fs,
        "mode_indices": vib_indices,
        "frequencies_cm1": freqs_cm1[vib_indices],
        "kinetic_eV": mode_kinetic,
        "potential_eV": mode_potential,
        "total_eV": mode_total,
        "total_vibrational_energy_eV": total_vibrational_energy,
        "relative_mode_energy": relative_mode_energy,
    }


def plot_molecular_mode_energies(mode_report: dict, max_modes: int = 6):
    times_fs = mode_report["times_fs"]
    mode_total = mode_report["total_eV"]
    frequencies_cm1 = mode_report["frequencies_cm1"]
    mode_indices = mode_report["mode_indices"]
    relative_mode_energy = mode_report["relative_mode_energy"]
    total_vibrational_energy = mode_report["total_vibrational_energy_eV"]

    ranking = np.argsort(mode_total.mean(axis=0))[::-1]
    selected = ranking[:min(max_modes, len(ranking))]
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    axes[0].plot(times_fs, total_vibrational_energy, color="black", linewidth=2.0, label="Total vibrational")
    for idx in selected:
        label = f"Mode {mode_indices[idx] - mode_indices[0]} ({frequencies_cm1[idx]:.0f} cm$^{{-1}}$)"
        axes[0].plot(times_fs, mode_total[:, idx], linewidth=1.3, label=label)
        axes[1].plot(times_fs, relative_mode_energy[:, idx], linewidth=1.3, label=label)
    axes[0].set_ylabel("Energy (eV)")
    axes[0].set_title("Molecular vibrational-mode energies")
    axes[0].legend(ncol=2, fontsize=9)
    axes[1].set_xlabel("Time (fs)")
    axes[1].set_ylabel("Fraction of total vib. energy")
    axes[1].set_title("Relative vibrational energy by mode")
    axes[1].legend(ncol=2, fontsize=9)
    fig.tight_layout()
    return fig


def plot_trajectory_snapshots(traj_path: Path, stride: int = 20, max_frames: int = 6):
    frames = read(traj_path, index=":")
    if not isinstance(frames, list):
        frames = [frames]
    sampled = frames[::stride][:max_frames]
    if not sampled:
        raise ValueError("No trajectory frames available for plotting.")

    fig, axes = plt.subplots(len(sampled), 2, figsize=(10, 3 * len(sampled)))
    if len(sampled) == 1:
        axes = np.array([axes])

    for i, atoms_i in enumerate(sampled):
        plot_atoms(atoms_i, axes[i, 0], rotation="0x,0y,0z", radii=0.5)
        plot_atoms(atoms_i, axes[i, 1], rotation="90x,0y,0z", radii=0.5)
        axes[i, 0].set_axis_off()
        axes[i, 1].set_axis_off()
        axes[i, 0].set_title(f"Frame {i * stride}: top")
        axes[i, 1].set_title(f"Frame {i * stride}: side")

    fig.tight_layout()
    return fig


def animate_trajectory(traj_path: Path, stride: int = 5, interval_ms: int = 120):
    frames = read(traj_path, index=":")
    if not isinstance(frames, list):
        frames = [frames]
    frames = frames[::stride]
    if not frames:
        raise ValueError("No trajectory frames available for animation.")

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    def draw_frame(i: int):
        for ax in axes:
            ax.clear()
            ax.set_axis_off()
        plot_atoms(frames[i], axes[0], rotation="0x,0y,0z", radii=0.5)
        plot_atoms(frames[i], axes[1], rotation="90x,0y,0z", radii=0.5)
        axes[0].set_title(f"Frame {i * stride}: top")
        axes[1].set_title(f"Frame {i * stride}: side")
        return axes

    ani = animation.FuncAnimation(fig, draw_frame, frames=len(frames), interval=interval_ms, blit=False)
    plt.close(fig)
    return HTML(ani.to_jshtml())

In [ ]:
poscar_path = choose_poscar(CONFIG["poscar_glob"], CONFIG["poscar_index"])
atoms = read(poscar_path, format="vasp")
ensure_velocities(atoms, CONFIG)
atoms.calc = build_mace_calculator(CONFIG)

summary = summarize_atoms(atoms)
print(f"Selected POSCAR: {poscar_path}")
print("Initial positions and velocities were read directly from this POSCAR.")
print(json.dumps(summary, indent=2))
print(f"Initial potential energy: {atoms.get_potential_energy():.6f} eV")
print(f"Initial max |force|: {np.abs(atoms.get_forces()).max():.6f} eV/A")

In [ ]:
plot_structure_views(atoms, title=f"Initial structure: {poscar_path.name}")
plt.show()

In [ ]:
history, traj_path, xyz_path = run_nve(atoms, CONFIG)
print(f"Trajectory written to: {traj_path}")
if xyz_path is not None:
    print(f"XYZ trajectory written to: {xyz_path}")
print(f"Final total energy: {history['etot_eV'][-1]:.6f} eV")
print(f"Initial total energy: {history['etot_eV'][0]:.6f} eV")
print(f"Energy drift: {history['etot_eV'][-1] - history['etot_eV'][0]:.6e} eV")

In [ ]:
molecule_ref = read(CONFIG["molecule_vasprun"], index=-1, format="vasp-xml")
molecule_natoms = len(molecule_ref)
plot_component_temperatures(
    traj_path,
    molecule_natoms=molecule_natoms,
    timestep_fs=CONFIG["timestep_fs"] * CONFIG["traj_write_interval"],
)
mode_basis = load_mode_basis(CONFIG["molecule_vasprun"], CONFIG["modes_npz"])
mode_report = analyze_molecular_vibrations(
    traj_path,
    mode_basis=mode_basis,
    molecule_natoms=molecule_natoms,
    timestep_fs=CONFIG["timestep_fs"] * CONFIG["traj_write_interval"],
    stride=CONFIG["mode_plot_stride"],
)
plot_molecular_mode_energies(mode_report, max_modes=CONFIG["mode_plot_max_modes"])
plt.show()

In [ ]:
plot_md_diagnostics(history)
plt.show()

In [ ]:
plot_trajectory_snapshots(traj_path, stride=CONFIG["snapshot_stride"])
plt.show()

In [ ]:
display(animate_trajectory(traj_path, stride=CONFIG["animation_stride"], interval_ms=CONFIG["animation_interval_ms"]))

In [ ]:
if nv is None:
    raise ImportError("nglview is not installed in this kernel. Install requirements.txt and restart the Jupyter kernel.")

interactive_frames = read(traj_path, index=":")
if not isinstance(interactive_frames, list):
    interactive_frames = [interactive_frames]

view = nv.show_asetraj(interactive_frames)
view.clear_representations()
view.add_unitcell()
view.add_ball_and_stick()
view.center()
view


## Interpretation

- Check the initial structure views for obvious overlaps or unrealistic molecule placement.
- Check whether the initial forces are excessively large.
- In short NVE, watch whether the structure immediately explodes, desorbs unphysically, or shows extreme energy drift.
- Use the animation to confirm that the incoming molecule moves with the expected initial velocity.
- Use the final `nglview` cell if you want to rotate and inspect the trajectory interactively with the mouse.
- If a configuration is suspicious, change `poscar_index` and rerun the notebook on another generated POSCAR.
